## Dataset Description: College Scorecard

The **College Scorecard** dataset from the U.S. Department of Education contains comprehensive data on U.S. colleges and universities, including admission rates, student demographics, costs, and outcomes.

### Task:
Predict whether an institution is **selective** (admission rate < 50%) based on institutional characteristics and student demographics.

### Key Columns:
- **SCHOOL_NAME**: Institution name
- **ADMISSION_RATE**: Overall admission rate (used to create target)
- **SAT_AVG**: Average SAT scores
- **STUDENT_SIZE**: Undergraduate enrollment
- **PREDOMINANT_DEGREE**: Predominant degree awarded
- **LOCALE**: Institution locale (city, suburb, town, rural)
- **PCT_WHITE/BLACK/HISPANIC/ASIAN**: Race/ethnicity demographic percentages
- **PCT_FEMALE**: Percentage of female students
- **PELL_RATE**: Percent receiving Pell Grants (socioeconomic proxy)
- **SELECTIVE**: Target variable (1: admission rate < 50%, 0: otherwise)

### Fairness Considerations:
Sensitive attributes include **race/ethnicity demographics**, **PCT_FEMALE** (gender), and **PELL_RATE** (socioeconomic status).

## 1. Data Downloading
Downloading College Scorecard data from the U.S. Department of Education API.

In [ ]:
import os
import pandas as pd
import numpy as np
import requests
from sklearn.model_selection import train_test_split
import time

DATASET_NAME = "college-scorecard"
API_KEY = "b2uiD1hLpLFGxckoJZ1TdcOdsmf5c5GQSeExcRTe"
BASE_URL = "https://api.data.gov/ed/collegescorecard/v1/schools"

OUTPUT_DIR = f"../../resources/datasets/{DATASET_NAME}"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Define fields to fetch
fields = [
    "id",
    "school.name",
    "school.state",
    "school.city",
    "school.locale",
    "school.degrees_awarded.predominant",
    "latest.admissions.admission_rate.overall",
    "latest.admissions.sat_scores.average.overall",
    "latest.student.size",
    "latest.student.demographics.race_ethnicity.white",
    "latest.student.demographics.race_ethnicity.black",
    "latest.student.demographics.race_ethnicity.hispanic",
    "latest.student.demographics.race_ethnicity.asian",
    "latest.student.demographics.race_ethnicity.aian",
    "latest.student.demographics.race_ethnicity.nhpi",
    "latest.student.demographics.female_share",
    "latest.aid.pell_grant_rate",
    "latest.cost.tuition.in_state",
    "latest.completion.completion_rate_4yr_150nt",
]

fields_str = ",".join(fields)

print("Fetching College Scorecard data via API...")
print("This will take 1-2 minutes to fetch institutions with admission data...")

# Fetch data with pagination
all_data = []
page = 0
per_page = 100

while True:
    params = {
        "api_key": API_KEY,
        "fields": fields_str,
        "per_page": per_page,
        "page": page,
        "latest.admissions.admission_rate.overall__range": "0..1",  # Only schools with admission rate data
    }
    
    response = requests.get(BASE_URL, params=params)
    
    if response.status_code != 200:
        print(f"Error: {response.status_code}")
        print(response.text)
        break
    
    data = response.json()
    results = data.get("results", [])
    
    if not results:
        break
    
    all_data.extend(results)
    print(f"Fetched page {page}: {len(results)} schools (Total: {len(all_data)})")
    
    page += 1
    time.sleep(0.1)  # Rate limiting courtesy
    
    # Safety limit
    if page > 100:
        break

print(f"\nTotal schools fetched: {len(all_data)}")

# Convert to DataFrame
df = pd.DataFrame(all_data)
print(f"Initial shape: {df.shape}")
print(f"Columns: {df.columns.tolist()[:10]}...")  # Show first 10

# Flatten nested column names (API returns dotted names)
df.columns = [col.replace('latest.', '').replace('school.', '').replace('admissions.', '').replace('student.', '').replace('demographics.', '').replace('aid.', '').replace('cost.', '').replace('completion.', '') for col in df.columns]

print(f"\nFlattened columns: {df.columns.tolist()}")

# Data cleaning
print("\nCleaning data...")

# Convert numeric columns
numeric_cols = [
    'admission_rate.overall',
    'sat_scores.average.overall',
    'size',
    'race_ethnicity.white',
    'race_ethnicity.black',
    'race_ethnicity.hispanic',
    'race_ethnicity.asian',
    'race_ethnicity.aian',
    'race_ethnicity.nhpi',
    'female_share',
    'pell_grant_rate',
    'tuition.in_state',
    'completion_rate_4yr_150nt',
    'degrees_awarded.predominant',
    'locale',
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Rename for clarity
rename_map = {
    'admission_rate.overall': 'ADMISSION_RATE',
    'sat_scores.average.overall': 'SAT_AVG',
    'size': 'STUDENT_SIZE',
    'race_ethnicity.white': 'PCT_WHITE',
    'race_ethnicity.black': 'PCT_BLACK',
    'race_ethnicity.hispanic': 'PCT_HISPANIC',
    'race_ethnicity.asian': 'PCT_ASIAN',
    'race_ethnicity.aian': 'PCT_AIAN',
    'race_ethnicity.nhpi': 'PCT_NHPI',
    'female_share': 'PCT_FEMALE',
    'pell_grant_rate': 'PELL_RATE',
    'tuition.in_state': 'TUITION',
    'completion_rate_4yr_150nt': 'COMPLETION_RATE',
    'degrees_awarded.predominant': 'PREDOMINANT_DEGREE',
    'name': 'SCHOOL_NAME',
    'state': 'STATE',
    'city': 'CITY',
    'locale': 'LOCALE',
}

df = df.rename(columns=rename_map)

# Filter to schools with admission rate data
df = df[df['ADMISSION_RATE'].notna()].copy()
print(f"After filtering for admission rate: {df.shape}")

# Create binary target: Selective (admission rate < 0.5)
df['SELECTIVE'] = (df['ADMISSION_RATE'] < 0.5).astype(int)

print(f"\nTarget distribution:")
print(df['SELECTIVE'].value_counts())
print(f"Selective rate: {df['SELECTIVE'].mean():.2%}")

# Select final features
feature_cols = [
    'SAT_AVG',
    'STUDENT_SIZE',
    'PCT_WHITE',
    'PCT_BLACK',
    'PCT_HISPANIC',
    'PCT_ASIAN',
    'PCT_FEMALE',
    'PELL_RATE',
    'TUITION',
    'COMPLETION_RATE',
    'PREDOMINANT_DEGREE',
    'LOCALE',
    'SELECTIVE'
]

available_features = [col for col in feature_cols if col in df.columns]
df_final = df[available_features].copy()

# Drop rows with too many missing values
df_final = df_final.dropna(subset=['SAT_AVG', 'STUDENT_SIZE'], how='any')

# Fill remaining missing values with median
for col in df_final.columns:
    if col not in ['SELECTIVE', 'SCHOOL_NAME']:
        if df_final[col].dtype in ['float64', 'int64']:
            df_final[col] = df_final[col].fillna(df_final[col].median())

print(f"\nAfter cleaning: {df_final.shape}")
print(f"Final columns: {df_final.columns.tolist()}")

# Split into Train/Test
train_df, test_df = train_test_split(
    df_final,
    test_size=0.2,
    random_state=42,
    stratify=df_final['SELECTIVE']
)

# Save
train_path = os.path.join(OUTPUT_DIR, "train.csv")
test_path = os.path.join(OUTPUT_DIR, "test.csv")

train_df.to_csv(train_path, index=False)
test_df.to_csv(test_path, index=False)

print(f"\nSaved train.csv to {train_path}")
print(f"Saved test.csv to {test_path}")
print(f"Train shape: {train_df.shape}, Test shape: {test_df.shape}")
print(f"\nSensitive attributes for fairness analysis:")
print(f"  - PCT_WHITE/BLACK/HISPANIC/ASIAN: Race/ethnicity percentages")
print(f"  - PCT_FEMALE: Gender representation")
print(f"  - PELL_RATE: Socioeconomic status (% receiving Pell Grants)")

Fetching College Scorecard data via API...
This will take 2-3 minutes to fetch ~7000 institutions...
Fetched page 0: 100 schools (Total: 100)
Fetched page 1: 100 schools (Total: 200)
Fetched page 2: 100 schools (Total: 300)
Fetched page 3: 100 schools (Total: 400)
Fetched page 4: 100 schools (Total: 500)
Fetched page 5: 100 schools (Total: 600)
Fetched page 6: 100 schools (Total: 700)
Fetched page 7: 100 schools (Total: 800)
Fetched page 8: 100 schools (Total: 900)
Fetched page 9: 100 schools (Total: 1000)
Fetched page 10: 100 schools (Total: 1100)
Fetched page 11: 100 schools (Total: 1200)
Fetched page 12: 100 schools (Total: 1300)
Fetched page 13: 100 schools (Total: 1400)
Fetched page 14: 100 schools (Total: 1500)
Fetched page 15: 100 schools (Total: 1600)
Fetched page 16: 100 schools (Total: 1700)
Fetched page 17: 100 schools (Total: 1800)
Fetched page 18: 100 schools (Total: 1900)
Fetched page 19: 46 schools (Total: 1946)

Total schools fetched: 1946
Initial shape: (1946, 19)
Colu